# FHIR R4 API ingestion: Raw and Bronze

Run this notebook once per resource type. The Fabric pipeline passes Patient, Encounter, Observation, and Condition through the resource_type parameter in the required order. The notebook follows the server-provided pagination link, stores the original Bundle response in Raw, logs request/save metadata, and merges distinct resource content versions into the Bronze Delta table.

Attach the schema-enabled Lakehouse lh_fhir_assignment as the default Lakehouse.

In [ ]:
# Fabric pipeline overrides resource_type for each ingestion activity.
resource_type = "Patient"
initial_lookback_days = 3
page_size = 100
overlap_minutes = 5
max_pages = 1000

In [ ]:
# Create the Medallion schemas in the attached, schema-enabled Lakehouse.
# This is safe to execute repeatedly and does not delete data.
for schema_name in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

print("Lakehouse schemas ready: bronze, silver, gold")
display(spark.sql("SHOW SCHEMAS"))

In [ ]:
import hashlib
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path

import requests
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

FHIR_BASE_URL = "https://hapi.fhir.org/baseR4"
SUPPORTED_RESOURCES = {"Patient", "Encounter", "Observation", "Condition"}
RAW_ROOT = Path("/lakehouse/default/Files/raw/fhir")
AUDIT_TABLE = "dbo.fhir_ingestion_audit"
WATERMARK_TABLE = "dbo.fhir_ingestion_watermarks"

# Normalize pipeline parameters, which can arrive as strings.
resource_type = str(resource_type).strip()
initial_lookback_days = int(initial_lookback_days)
page_size = int(page_size)
overlap_minutes = int(overlap_minutes)
max_pages = int(max_pages)

if resource_type not in SUPPORTED_RESOURCES:
    raise ValueError(f"resource_type must be one of {sorted(SUPPORTED_RESOURCES)}")
if initial_lookback_days < 1 or page_size < 1 or max_pages < 1 or overlap_minutes < 0:
    raise ValueError("Lookback, page size, and max pages must be positive; overlap cannot be negative.")

spark.conf.set("spark.sql.session.timeZone", "UTC")
for schema_name in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")


def utc_now():
    return datetime.now(timezone.utc)


def parse_utc(value):
    if value is None or str(value).strip() == "":
        return None
    value = str(value).strip()
    if value.endswith("Z"):
        value = value[:-1] + "+00:00"
    parsed = datetime.fromisoformat(value)
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc)


def table_exists(table_name):
    return spark.catalog.tableExists(table_name)


audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("resource_type", StringType(), False),
    StructField("page_number", IntegerType(), False),
    StructField("api_call_timestamp", StringType(), True),
    StructField("extraction_timestamp", StringType(), True),
    StructField("saved_timestamp", StringType(), True),
    StructField("api_url_or_params", StringType(), True),
    StructField("raw_file_path", StringType(), True),
    StructField("metadata_file_path", StringType(), True),
    StructField("http_status", IntegerType(), True),
    StructField("records_received", IntegerType(), True),
    StructField("page_status", StringType(), True),
    StructField("run_status", StringType(), True),
    StructField("error_message", StringType(), True)
])


def persist_audit(rows):
    if not rows:
        return
    audit_df = spark.createDataFrame(rows, schema=audit_schema)
    if not table_exists(AUDIT_TABLE):
        audit_df.write.format("delta").mode("overwrite").saveAsTable(AUDIT_TABLE)
    else:
        target = DeltaTable.forName(spark, AUDIT_TABLE)
        condition = (
            "t.run_id = s.run_id AND "
            "t.resource_type = s.resource_type AND "
            "t.page_number = s.page_number"
        )
        (target.alias("t").merge(audit_df.alias("s"), condition)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())


# Read this resource's latest successful source watermark.
run_started = utc_now()
run_id = run_started.strftime("%Y%m%dT%H%M%S%fZ")
extraction_date = run_started.strftime("%Y-%m-%d")
bronze_table = f"bronze.{resource_type}"
previous_watermark = None

if table_exists(WATERMARK_TABLE):
    rows = (spark.table(WATERMARK_TABLE)
            .filter(F.col("resource_type") == resource_type)
            .limit(1).collect())
    if rows:
        previous_watermark = rows[0]["last_successful_source_updated_utc"]

if previous_watermark:
    cutoff_dt = parse_utc(previous_watermark) - timedelta(minutes=overlap_minutes)
    cutoff_reason = "saved source watermark with overlap"
else:
    cutoff_dt = run_started - timedelta(days=initial_lookback_days)
    cutoff_reason = f"initial lookback of {initial_lookback_days} days"

cutoff_text = cutoff_dt.strftime("%Y-%m-%dT%H:%M:%SZ")
print("Run ID:", run_id)
print("Resource:", resource_type)
print("Incremental cutoff UTC:", cutoff_text, f"({cutoff_reason})")
print("Page size:", page_size, "Max pages:", max_pages)

bronze_schema = StructType([
    StructField("resource_type", StringType(), False),
    StructField("resource_id", StringType(), False),
    StructField("source_last_updated", StringType(), True),
    StructField("resource_json", StringType(), False),
    StructField("content_hash", StringType(), False),
    StructField("extraction_date", StringType(), True),
    StructField("extraction_timestamp", StringType(), True),
    StructField("api_url_or_params", StringType(), True),
    StructField("raw_file_path", StringType(), False),
    StructField("ingestion_run_id", StringType(), False),
    StructField("saved_timestamp", StringType(), True)
])

session = requests.Session()
session.headers.update({"Accept": "application/fhir+json"})
current_url = f"{FHIR_BASE_URL}/{resource_type}"
current_params = {
    "_count": page_size,
    "_format": "json",
    "_lastUpdated": f"ge{cutoff_text}"
}

page_count = 0
records = []
audit_records = []
current_call_timestamp = None
current_actual_url = None
current_http_status = 0

try:
    while current_url and page_count < max_pages:
        current_call_timestamp = utc_now().isoformat()
        response = session.get(current_url, params=current_params, timeout=90)
        current_http_status = int(response.status_code)
        current_actual_url = response.url
        response.raise_for_status()

        bundle = response.json()
        if bundle.get("resourceType") != "Bundle" or bundle.get("type") != "searchset":
            raise ValueError(
                f"Expected a FHIR searchset Bundle; received "
                f"resourceType={bundle.get('resourceType')!r}, type={bundle.get('type')!r}"
            )

        page_count += 1
        entries = bundle.get("entry", []) or []
        raw_dir = (
            RAW_ROOT / f"resource={resource_type}"
            / f"extraction_date={extraction_date}"
            / f"run_id={run_id}"
        )
        raw_dir.mkdir(parents=True, exist_ok=True)
        raw_path = raw_dir / f"page_{page_count:05d}.json"
        metadata_path = raw_dir / f"page_{page_count:05d}.metadata.json"

        # Preserve the original response body as received. Metadata is a separate sidecar.
        raw_path.write_bytes(response.content)
        saved_timestamp = utc_now().isoformat()
        metadata = {
            "run_id": run_id,
            "resource_type": resource_type,
            "page_number": page_count,
            "api_call_timestamp": current_call_timestamp,
            "extraction_timestamp": current_call_timestamp,
            "saved_timestamp": saved_timestamp,
            "api_url_or_params": current_actual_url,
            "raw_file_path": str(raw_path),
            "http_status": current_http_status,
            "records_received": len(entries),
            "incremental_cutoff_utc": cutoff_text
        }
        metadata_path.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        audit_records.append({
            "run_id": run_id,
            "resource_type": resource_type,
            "page_number": page_count,
            "api_call_timestamp": current_call_timestamp,
            "extraction_timestamp": current_call_timestamp,
            "saved_timestamp": saved_timestamp,
            "api_url_or_params": current_actual_url,
            "raw_file_path": str(raw_path),
            "metadata_file_path": str(metadata_path),
            "http_status": current_http_status,
            "records_received": len(entries),
            "page_status": "RAW_SAVED",
            "run_status": "IN_PROGRESS",
            "error_message": None
        })

        for entry in entries:
            resource = entry.get("resource") or {}
            if resource.get("resourceType") != resource_type:
                continue
            resource_id = resource.get("id")
            if not resource_id:
                continue

            resource_json = json.dumps(
                resource, sort_keys=True, separators=(",", ":"), ensure_ascii=False
            )
            content_hash = hashlib.sha256(resource_json.encode("utf-8")).hexdigest()
            records.append({
                "resource_type": resource_type,
                "resource_id": str(resource_id),
                "source_last_updated": resource.get("meta", {}).get("lastUpdated"),
                "resource_json": resource_json,
                "content_hash": content_hash,
                "extraction_date": extraction_date,
                "extraction_timestamp": current_call_timestamp,
                "api_url_or_params": current_actual_url,
                "raw_file_path": str(raw_path),
                "ingestion_run_id": run_id,
                "saved_timestamp": saved_timestamp
            })

        current_url = next(
            (item.get("url") for item in bundle.get("link", [])
             if item.get("relation") == "next"),
            None
        )
        # The server-provided next link already has its pagination parameters.
        current_params = None

    if current_url:
        raise RuntimeError(
            f"Pagination safety limit reached ({max_pages} pages). "
            "The run is marked failed and the watermark is not advanced."
        )

    source_df = (
        spark.createDataFrame(records, schema=bronze_schema)
        if records else spark.createDataFrame([], schema=bronze_schema)
    )

    if not table_exists(bronze_table):
        source_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
    elif records:
        source_df = source_df.dropDuplicates(
            ["resource_type", "resource_id", "content_hash"]
        )
        target = DeltaTable.forName(spark, bronze_table)
        merge_condition = (
            "t.resource_type = s.resource_type AND "
            "t.resource_id = s.resource_id AND "
            "t.content_hash = s.content_hash"
        )
        (target.alias("t").merge(source_df.alias("s"), merge_condition)
         .whenNotMatchedInsertAll().execute())

    source_times = [
        parse_utc(record["source_last_updated"])
        for record in records
        if record.get("source_last_updated")
    ]
    source_times = [value for value in source_times if value is not None]

    # If a complete scan returned no matching resources, advance conservatively
    # to run-start minus the overlap. The next run also subtracts the overlap,
    # leaving a safety window for source/indexing delays. If records were
    # returned but have no source timestamp, keep the prior watermark.
    if records and len(source_times) == len(records):
        watermark_candidate = max(source_times)
    elif not records:
        watermark_candidate = run_started - timedelta(minutes=overlap_minutes)
        print("No resources matched the completed scan; advancing to the scanned time boundary.")
    else:
        # Do not advance past resources with missing source timestamps.
        watermark_candidate = None
        print("One or more resources lacked source lastUpdated values; existing watermark retained.")

    for item in audit_records:
        item["run_status"] = "SUCCESS"
    persist_audit(audit_records)

    if watermark_candidate is not None:
        latest_source_time = watermark_candidate.isoformat(timespec="microseconds")
        watermark_schema = StructType([
            StructField("resource_type", StringType(), False),
            StructField("last_successful_source_updated_utc", StringType(), False),
            StructField("last_successful_run_id", StringType(), False),
            StructField("watermark_updated_at_utc", StringType(), False)
        ])
        watermark_df = spark.createDataFrame([{
            "resource_type": resource_type,
            "last_successful_source_updated_utc": latest_source_time,
            "last_successful_run_id": run_id,
            "watermark_updated_at_utc": utc_now().isoformat()
        }], schema=watermark_schema)

        if not table_exists(WATERMARK_TABLE):
            watermark_df.write.format("delta").mode("overwrite").saveAsTable(WATERMARK_TABLE)
        else:
            watermark_target = DeltaTable.forName(spark, WATERMARK_TABLE)
            (watermark_target.alias("t").merge(
                watermark_df.alias("s"),
                "t.resource_type = s.resource_type"
            ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

    print("Ingestion completed successfully.")
    print("Pages saved:", page_count)
    print("Resources observed:", len(records))
    print("Bronze table:", bronze_table)
    print("Audit table:", AUDIT_TABLE)
    print("Watermark table:", WATERMARK_TABLE)

except Exception as exc:
    failure_record = {
        "run_id": run_id,
        "resource_type": resource_type,
        "page_number": page_count + 1,
        "api_call_timestamp": current_call_timestamp or utc_now().isoformat(),
        "extraction_timestamp": current_call_timestamp or utc_now().isoformat(),
        "saved_timestamp": None,
        "api_url_or_params": current_actual_url or current_url,
        "raw_file_path": None,
        "metadata_file_path": None,
        "http_status": current_http_status,
        "records_received": 0,
        "page_status": "RUN_FAILURE",
        "run_status": "FAILED",
        "error_message": str(exc)[:4000]
    }
    for item in audit_records:
        item["run_status"] = "FAILED"
    audit_records.append(failure_record)
    try:
        persist_audit(audit_records)
    except Exception as audit_error:
        print("Could not persist audit failure details:", str(audit_error))
    raise
finally:
    session.close()